In [ ]:
import pandas as pd
import numpy as np

from 

.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score,mean_squared_error



In [ ]:
#cargar dataset
df=pd.read_csv(r"E:\Proyectos\GIT\machine-learning-intep\ventas_ml_clase2.csv")
df.head()

In [ ]:
print(f"filas: {df.shape[0]}, columnas: {df.shape[1]}")
print("")

print("Tipos de datos:")
for col, type in df.dtypes.items():
    print(f"{col}: {type}")

In [ ]:
df.describe(include="all").transpose().head()

In [ ]:
# predecir las ventas usando variables: marketing, precio, temporada, tienda, canal.
X = df[["marketing","precio","temporada","tienda","canal"]]
Y = df["ventas"]
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)
print(f"Datos de entrenamiento: {X_train.shape[0]} filas ({X_train.shape[0]/len(df)*100:.0f}% del dataset)")
print(f"Datos de prueba: {X_test.shape[0]} filas ({X_test.shape[0]/len(df)*100:.0f}% del total)")
print()
print("El modelo aprenderá con el 80% de los datos y se evaluara con el 20% restante:")
print("Esto evitara que el modelo memorice los datos y permita evaluar su capacidad de generación a datos nuevos.")

In [ ]:
from sklearn.linear_model import LinearRegression


numeric_features = ["marketing", "precio", "temporada"]
categorical_features = ["tienda", "canal"]

preprocess = ColumnTransformer(
    transformers=(
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    )
)

model = LinearRegression()

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", model)])

pipe

In [ ]:
pipe.fit(X_train, y_train) # entrnamos los datos de entrenamiento

In [ ]:
y_pred = pipe.predict(X_test) # generar predicciones con los dato de prueba

mae=mean_absolute_error(y_test, y_pred) # Error absoluto promeido
rmse=np.sqrt(mean_squared_error(y_test, y_pred)) # Raiz del error cuadratico medio
r2=r2_score(y_test, y_pred) # r2: propoción de varianza explicada por el modelo

media_ventas = y_test.mean() # promerio de ventas reales
print(f"MAE ( Error absoluto promedio): {mae:.2f}")
print(f"RMSE ( Raiz del error cuadratico medio): {rmse:.2f}")
print(f"R2 ( Propoción de varianza explicada por el modelo): {r2:.4f}({r2*100:.1f}%) ")
print("="*50)
print("Media de venta reales:",f"${media_ventas:.2f}")
print("Error relativo (MAE/Media):",f"{(mae/media_ventas)*100:.1f}%")

print ("="*50)
print("Interpretación de la metricas:")
print(f"en promerio, el modelo se equivoca en ${mae:.2f} por producción.")
print(f"esto reprenta un {(mae/media_ventas)*100:.1f}% del promedio de ventas reales.")
print(f"el modelo explica {r2*100:.1f}% de la varianza en las ventas,lo cual es bastante bueno para un modelo simple")
print(f"el{(1-r2)*100:.1f}% restante se debe a factores no incluidos en el modulo")

if r2 < 0.8:
    print("Valorazión: buen ajuste para un modelo lineal")
elif r2 >= 0.5:
    print("valorización:Ajuste moderado. hay espacio para mejorar con modelos")
    print("Mas complejos o incluyendo mas variables")
else:
    print("Valoración: Ajuste bajo. se recomienda revisar las variables o porbar otros modelos")
    
          

In [ ]:
import matplotlib.pyplot as plt      
# --- Gráfica de Ventas Reales vs Predichas con línea de tendencia ---
fig, ax = plt.subplots(figsize=(8, 6))

# Cada punto es una observación del set de prueba
# Eje X = venta real, Eje Y = venta predicha por el modelo
ax.scatter(y_test, y_pred, alpha=0.5, edgecolors="steelblue",
           facecolors="lightblue", linewidths=0.8, label="Predicciones")

# Línea de predicción perfecta (donde real = predicho)
# Si el modelo fuera perfecto, todos los puntos estarían sobre esta línea
limite_min = min(y_test.min(), y_pred.min()) - 10
limite_max = max(y_test.max(), y_pred.max()) + 10
ax.plot([limite_min, limite_max], [limite_min, limite_max],
        color="red", linestyle="--", linewidth=1.5,
        label="Prediccion perfecta (real = predicho)")

# Etiquetas y título
ax.set_xlabel("Ventas reales ($)", fontsize=12)
ax.set_ylabel("Ventas predichas ($)", fontsize=12)
ax.set_title(f"Ventas reales vs predichas  (R² = {r2:.2f})", fontsize=14)
ax.legend(fontsize=10)

# Ajustar los ejes para que tengan la misma escala
ax.set_xlim(limite_min, limite_max)
ax.set_ylim(limite_min, limite_max)
ax.set_aspect("equal")

plt.tight_layout()
plt.show()


# coefientes del modelo : que variables pesan mas?
un modelo de regresión lineal es una formula de ***suma pondera***

ventas= base + (peso1 x marketing) + (peso 2 x precio) + (peso3 x temporada) +...

In [ ]:
# Extraer los coeficientes (peso) que el modelo aprendió

# Obtener los nombres de las columnas categóricas después del OneHotEncoding
ohe = pipe.named_steps["preprocess"].named_transformers_["cat"]
cat_cols = ohe.get_feature_names_out(["tienda", "canal"]).tolist()

# Lista completa de variables: numericas  + las generadas por el OneHotEncoding
feature_names = numeric_features + cat_cols

# Los coeficientes son los "pesos" de la fórmula:
# ventas = intercepto + coef1*marketing + coef2*precio + ... + coefN*canal_online
coef = pipe.named_steps["model"].coef_  # Coeficientes de cada variable



# Organizar de mayor a menor
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": coef
}).sort_values(by="coef",  ascending=False)

# Mostrar la tabla de coeficientes

print("="*50)
print("Coeficientes del modelo")
for _, fila in coef_df.iterrows():
    signo = "+" if fila["coef"] > 0 else ""
    barra = "^" if fila["coef"] > 0 else "v"
    print(f" {barra} {fila['feature']:20s} {signo}{fila['coef']:.4f}")


In [ ]:
row = X_test.iloc[[0]].copy()  # Tomamos la primera fila del set de prueba para hacer una predicción individual
print("datos de la observación seleccionada:")
print(row.to_string(index=False))

In [ ]:
pred_base = float(pipe.predict(row)[0])  # Predicción del modelo para esa fila

row_more_marketing = row.copy()
row_more_marketing["marketing"] = row_more_marketing["marketing"] * 1.10  # Aumentamos el gasto en marketing en 10%

pred_more_marketing = float(pipe.predict(row_more_marketing)[0])  # Predicción con más marketing

# -- calcular la diferencia
mkt_original = float(row["marketing"].values[0])
mkt_nuevo = float(row_more_marketing["marketing"].values[0])
diferencia = pred_more_marketing - pred_base

print("="*50)
print(f"inversion marketing original: ${mkt_original:.2f}")
print(f"inversion marketing (10%):    ${mkt_nuevo:.2f}")
print(f"aumento de la inversion: ${mkt_nuevo - mkt_original:.2f}")
print("="*50)

print(f"predicción de ventas (base): ${pred_base:.2f}")
print(f" predicción de ventas (+10%): ${pred_more_marketing:.2f}")
print(f" aumento en ventas por +10% marketing: ${diferencia:.2f}")
print("="*50)

